In [ ]:
#Step-1
#Loads the Pandas library into Python's memory so one can access its spreadsheet-like functions.
try:
    import pandas as pd
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install pandas
    import pandas as pd


# Load dataset using a verified raw GitHub URL
url = "https://raw.githubusercontent.com/SubhrojitPatra/network-latency-analyzer/refs/heads/main/network_latency.csv"

# Read dataset into cloud memory
df = pd.read_csv(url)

#Display total count and preview top 5 rows
print(f"Database loaded successfully! Total records: {len(df)}")
display(df.head(5))

In [ ]:
#Step-2
#Display the column names of the DataFrame object
df.info()

In [ ]:
#step-3
"""Column headers with spaces or capital letters (like "Revenue (Millions)") cause syntax errors.
df.columns.str.replace(' ', '_'): Replaces all spaces with underscores.
.str.lower(): Converts all column headers to lowercase text so you never have to guess capitalization.
"""

df.columns = df.columns.str.replace(' ','_').str.lower()
print(df.columns.tolist())

In [ ]:
#step-4
"""
df.isnull(): Scans every cell and marks empty cells as True and filled cells as False.

.sum(): Counts up all the True values for each column so you can see where data is missing.
"""

print(df.isnull().sum())

In [ ]:
#step-5
"""
df.dropna(subset=[...]): Removes only the rows where values in revenue_(millions) or metascore are missing, saving the cleaned result into a new variable called df_clean.

Why assign to df_clean: It keeps your original dataset (df) untouched in memory while giving you a fresh table ready for analysis.
"""

df_clean = df.dropna(subset=['latency_ms'])
print(f"Cleaned rows: {len(df_clean)} (dropped {len(df)-len(df_clean)} empty rows)")

In [ ]:
"""
After completing Data ingestion and cleaning now we can start analyzing the data, and the first step is exploratory data analysis.
Here we will try to answer some basic questions from our database...
"""

# Set of questions to answer during the analysis

"""
A. Performance
What is the average, median, minimum and maximum latency?
What does the overall latency distribution look like?
B. Reliability
What is the success rate?
What is the failure rate?
C. Time behavior
How does latency change over time?
When do latency spikes occur?
D. Stability
How consistent is the network latency?
Are there significant outliers?
E. Failures
When do request failures occur?
Is there any observable relationship between high latency and failures?
"""


In [ ]:
#Step-6
# A. Performance
#What is the average, median, minimum and maximum latency?

#.aag(): Aggregates the latency_ms column to calculate the mean, median, min and max values.
latency_summary = df_clean['latency_ms'].agg(mean = "mean"
                             ,median = "median"
                             ,min = "min"
                             ,max = "max")
latency_summary

In [ ]:
#Step-7
#What does the overall latency distribution look like?

# why try-except block? Because matplotlib is not a default library in Python, so we need to install it first if it is not already installed.
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    !{sys.executable} -m pip install matplotlib
    import matplotlib.pyplot as plt
import numpy as np
import sys

#What we try to find out here is the distribution of latency values in the dataset. We can visualize this using a histogram, which will show us how frequently different latency values occur.
x = df_clean['latency_ms']
plt.hist(x,bins=np.arange(0, x.max(), 50), edgecolor='black')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.title('Latency Distribution')
plt.show()

In [ ]:
#Step-8
"""
B. Reliability
What is the success rate?
"""

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    !{sys.executable} -m pip install matplotlib
    import matplotlib.pyplot as plt
import numpy as np
import sys

#what we use in y is the value_counts() function to count the occurrences of each unique value in the "status" column. We then use reindex() to ensure that the order of the values is consistent (success first, then failure) and fill_value=0 to handle any missing values. Finally, we convert the result to a NumPy array using to_numpy() for plotting.
y = df["status"].value_counts().reindex(["success", "failure"], fill_value=0).to_numpy()
mylabels = ['Success', 'Failure']
plt.pie(y, labels=mylabels)
plt.title('Success Rate')
plt.show()

#Step-9
#What is the failure rate?
#faliure rate is the complement of success rate, so we can calculate it as follows:
success_count = df["status"].value_counts().get("success", 0)
failure_count = df["status"].value_counts().get("failure", 0)
total_count = success_count + failure_count
failure_rate = (failure_count / total_count) * 100 if total_count > 0 else 0
print(f"Failure Rate: {failure_rate:.2f}%")

In [ ]:

#Step-10
#C. Time behavior
#How does latency change over time?

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    !{sys.executable} -m pip install matplotlib
    import matplotlib.pyplot as plt
import numpy as np
import sys

#define x and y for plotting, where x is the latency values and y is the corresponding timestamps. We will convert the timestamps to datetime values and sort them chronologically to ensure that the plot accurately reflects the time series data.
x = df_clean["latency_ms"]
y = df_clean["timestamp"]

# Convert timestamps to datetime values and sort chronologically
plot_data = pd.DataFrame({
    "timestamp": pd.to_datetime(y),
    "latency_ms": x
}).sort_values("timestamp")

plt.figure(figsize=(12, 5))
plt.plot(
    plot_data["timestamp"],
    plot_data["latency_ms"],
    color="steelblue",
    linewidth=1.5,
    marker="o",
    markersize=3
)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.xlabel('Timestamp')
plt.ylabel('Latency (ms)')
plt.title('Latency Over Time')
plt.show()


#When do latency spikes occur?




In [ ]:
#step-11
"""
D. Stability
How consistent is the network latency?

The median latency is about 116 ms, with Q1 = 113 ms and Q3 = 120 ms, giving an IQR of only 7 ms.
This means most latency values stay tightly clustered around the median, indicating generally stable network performance.
A small number of outliers still exist, which explains the wider overall range seen in the dataset.
"""

# Quick stability check using the cleaned latency data
latency = df_clean["latency_ms"]
q1 = latency.quantile(0.25)
median = latency.median()
q3 = latency.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = latency[(latency < lower_bound) | (latency > upper_bound)]

print(f"Median latency: {median:.2f} ms")
print(f"Q1: {q1:.2f} ms | Q3: {q3:.2f} ms | IQR: {iqr:.2f} ms")
print(f"Outlier count: {len(outliers)}")
print("Conclusion: network latency is fairly consistent for most observations, with occasional spikes that indicate instability.")
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    !{sys.executable} -m pip install matplotlib
    import matplotlib.pyplot as plt
import numpy as np
import sys


# Stability check: visualize how tightly the latency values are clustered
# and whether there are extreme outliers that can affect reliability.

latency = df_clean["latency_ms"]

plt.figure(figsize=(8, 5))
plt.boxplot(
    latency,
    patch_artist=True,
    boxprops=dict(facecolor="lightsteelblue", edgecolor="navy"),
    medianprops=dict(color="darkorange", linewidth=2),
    whiskerprops=dict(color="navy"),
    capprops=dict(color="navy")
)
plt.title("Network Latency Stability")
plt.ylabel("Latency (ms)")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

